# Smart Logistics IoT Simulation
**Course:** MO-IT148 — Application Development and Emerging Technologies <br>
**Week:** 2 — IoT Data Simulation <br>
**Group:** [Group Name] <br>
**Description:** Simulates raw sensor data (GPS, RFID, temperature) for a smart logistics tracking system. Generates Sensor Readings first, then derives the Shipment Registry from RFID tags. Exports to CSV for downstream processing.<br>

## 1. Setup
Imports and global constants. All counts and bounds live here — generators read from these, never hard-code their own.

In [6]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

# Simulation scale
NUM_SHIPMENTS = 10
READINGS_PER_SHIPMENT = 5

# Coordinate bounds (Philippines region)
LAT_MIN, LAT_MAX = 4.0, 21.0
LON_MIN, LON_MAX = 116.0, 127.0

# Temperature range (Celsius)
TEMP_MIN, TEMP_MAX = 15.0, 35.0

# Base timestamp — readings increment from here
START_TIMESTAMP = datetime(2025, 1, 1, 8, 0, 0)

## 2. Sensor Readings
Generates raw sensor data rows. Each row represents one sensor snapshot for a shipment in transit. RFID tags are created here and used as the shipment identifier throughout.

In [7]:
rows = []

for shipment_index in range(NUM_SHIPMENTS):
    rfid_tag = f"RFID-{shipment_index + 1:03d}"
    # rfid_tag generated once per shipment — shared across all its readings

    for reading_index in range(READINGS_PER_SHIPMENT):
        reading_id = f"READ-{shipment_index * READINGS_PER_SHIPMENT + reading_index + 1:03d}"
        # reading_id is unique across the entire dataset, not just per shipment

        timestamp = START_TIMESTAMP + timedelta(hours=shipment_index * READINGS_PER_SHIPMENT + reading_index)
        # each reading increments by 1 hour from START_TIMESTAMP

        rows.append({
            "reading_id": reading_id,
            "rfid_tag": rfid_tag,
            "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
            "latitude": round(np.random.uniform(LAT_MIN, LAT_MAX), 6),
            "longitude": round(np.random.uniform(LON_MIN, LON_MAX), 6),
            "temperature_c": round(np.random.uniform(TEMP_MIN, TEMP_MAX), 2),
        })

sensor_readings_df = pd.DataFrame(rows)
print(f"Sensor Readings: {len(sensor_readings_df)} rows")

Sensor Readings: 50 rows


## 3. Shipment Registry
Built from the RFID tags extracted from Sensor Readings. One row per unique shipment. Never generates its own RFID tags — reads them from Readings.

In [8]:
GOODS_CATEGORIES = ["Dry Goods", "Perishable", "Electronics", "Clothing", "Industrial"]
CITIES = ["Manila", "Cebu", "Davao", "Quezon City", "Zamboanga", "Taguig", "Pasig", "Cagayan de Oro"]

registry_rows = []

for vehicle_index, rfid_tag in enumerate(sensor_readings_df["rfid_tag"].unique()):
    # enumerate() gives both the position (vehicle_index) and the value (rfid_tag)
    # .unique() extracts one rfid_tag per shipment — no duplicates

    origin = random.choice(CITIES)
    destination = random.choice([city for city in CITIES if city != origin])
    # destination must differ from origin — list comprehension filters origin out

    registry_rows.append({
        "rfid_tag": rfid_tag,
        "goods_category": random.choice(GOODS_CATEGORIES),
        "origin": origin,
        "destination": destination,
        "package_count": random.randint(1, 50),
        "vehicle_id": f"VH-{vehicle_index + 1:03d}",
        "driver_id": f"DR-{vehicle_index + 1:03d}",
    })

shipment_registry_df = pd.DataFrame(registry_rows)
print(f"Shipment Registry: {len(shipment_registry_df)} rows")

Shipment Registry: 10 rows


## 4. Export
Writes the generated data to CSV (and JSON if required). Run this before Preview.

In [9]:
sensor_readings_df.to_csv("sensor_readings.csv", index=False)
shipment_registry_df.to_csv("shipment_registry.csv", index=False)

# TODO: export to JSON (pending mentoring confirmation)

print("Files written:")
print("  → sensor_readings.csv")
print("  → shipment_registry.csv")

Files written:
  → sensor_readings.csv
  → shipment_registry.csv


In [10]:
print(f"""
Simulation Summary
------------------
Shipments:              {NUM_SHIPMENTS}
Readings per shipment:  {READINGS_PER_SHIPMENT}
Total readings:         {len(sensor_readings_df)}
Registry entries:       {len(shipment_registry_df)}

Files written:
  → sensor_readings.csv
  → shipment_registry.csv
""")


Simulation Summary
------------------
Shipments:              10
Readings per shipment:  5
Total readings:         50
Registry entries:       10

Files written:
  → sensor_readings.csv
  → shipment_registry.csv



## 5. Preview
Prints sample rows from the exported data to verify output looks correct.

In [11]:
print("=== Sensor Readings (first 5 rows) ===")
display(sensor_readings_df.head())

print("=== Shipment Registry (first 5 rows) ===")
display(shipment_registry_df.head())

=== Sensor Readings (first 5 rows) ===


,reading_id,rfid_tag,timestamp,latitude,longitude,temperature_c
0,READ-001,RFID-001,2025-01-01 08:00:00,8.942282,119.589392,26.45
1,READ-002,RFID-001,2025-01-01 09:00:00,6.182206,116.509235,31.90
2,READ-003,RFID-001,2025-01-01 10:00:00,17.298876,123.862261,31.04
3,READ-004,RFID-001,2025-01-01 11:00:00,9.387570,116.069630,25.17
4,READ-005,RFID-001,2025-01-01 12:00:00,16.422941,125.514495,26.49


=== Shipment Registry (first 5 rows) ===


,rfid_tag,goods_category,origin,destination,package_count,vehicle_id,driver_id
0,RFID-001,Dry Goods,Pasig,Zamboanga,24,VH-001,DR-001
1,RFID-002,Electronics,Davao,Cebu,40,VH-002,DR-002
2,RFID-003,Perishable,Zamboanga,Quezon City,21,VH-003,DR-003
3,RFID-004,Perishable,Cebu,Taguig,10,VH-004,DR-004
4,RFID-005,Clothing,Zamboanga,Manila,39,VH-005,DR-005
